In [19]:
import os
import json
import glob
import networkx as nx
import numpy as np
import pandas as pd

In [ ]:
## The steps that were taken up to here were: 

#### - Cloned the GitRepo
#### - Took my X account's bearer token and placed it into a json file in accordance to the directions in the repo.
#### - Added $25 in credits to rehydrate the tweets in this data set using the "tweet_rehydrator.py" script.
#### - This script ran and rehydrated tweets until the credits ran out.
#### - The resulting subset of data that was placed into the "rehydrated_tweet_data" folder was used when executing "graph_maker.py" script.
#### - This yields a directed graph data to use for my custom analysis for project 1. 
#### - The repository code shows that there should be edgelist files generated after running the "graph_maker.py" script.

In [2]:
### Checking on this last point.
candidates = glob.glob("**/*.edgelist", recursive=True)
candidates_sorted = sorted(candidates, key=lambda p: os.path.getmtime(p), reverse=True)
## looking at the edges and the weighted edges optiosn, as both are generated by the code used in this dataset. 
candidates_sorted

['network.weighted.edgelist', 'congress.edgelist']

In [ ]:
### Using this generated edgelist in order to begin working with the subset data network graph.
### Specifically choosing the edgelist, because the project looks solely at the degree centrality and the eigenvector centrality.
### I dont want any weights interfering with the raw calculation. However, i do want to acknowledge that the centrality measures, specifically, 
### eigenvector centrality cabn be calculated with the weighted edges. This would give additional nuance to pwoer within a network, building on the 
### authors previous work.

#### Using the data that is directly from the Repository, no new pulls needed. 

In [3]:
G = nx.read_edgelist('congress.edgelist', create_using=nx.DiGraph(), data=True )
print("Loaded: 'congress.edgelist'")
print("Directed:", G.is_directed())
print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())

Loaded: 'congress.edgelist'
Directed: True
Nodes: 475 Edges: 13289


In [4]:
## Getting the needed metrics for the eventual centrality calculations 
density = nx.density(G)
print("Density:", density)

## Do we have any isolated nodes? 
isolated_nodes = list(nx.isolates(G))
print("Count of isolated nodes:", len(isolated_nodes))

## Looking at the nodes in the graph that are weakly connected. S

Density: 0.05902287363979569
Count of isolated nodes: 0


In [5]:
## Seeing how much of these connections are weakly connected.
wccs = list(nx.weakly_connected_components(G))
print("Weakly connected components:", len(wccs))

largest_wcc = max(wccs, key=len)
print("Largest WCC size:", len(largest_wcc))

Weakly connected components: 1
Largest WCC size: 475


In [37]:
# Ok so the largest weakly connected component is the same size of the entire subset network, so there are no "island" nodes. 
# THere is 1 weakly connected component, so its literally just the network itself. 
# This is good for the eigenvector centrality calculations, no subgraph extractions needed. 

In [6]:
## Calculcating the in degree centrality for the G chart (no calculated weights)
in_deg_cent = nx.in_degree_centrality(G)

## Calculcating the out degree centrality for the G chart (no calculated weights)
out_deg_cent = nx.out_degree_centrality(G)


In [7]:
## Gettign the Eigenvector centrality, reversing the G network to look at those nodes that are spoken to, not who they are just speaking at.
eig = nx.eigenvector_centrality(G.reverse(), max_iter=5000, tol=1e-8)
# eig

#### Looking at the initial list results 

In [8]:
### Gettign the top 10 values dfor each of these metrics
print("Top 10 in-degree Centrality Scores:", sorted(in_deg_cent.items(), key=lambda x: x[1], reverse=True)[:10])
print("Top 10 out-degree Centrality Scores:", sorted(out_deg_cent.items(), key=lambda x: x[1], reverse=True)[:10])
print ('-'*100)

### Placing the values in a data fram
centrality_df = pd.DataFrame({
    "node": list(G.nodes()),
    "in_deg_cent": [in_deg_cent[n] for n in G.nodes()],
    "out_deg_cent": [out_deg_cent[n] for n in G.nodes()],
    "eigenvector": [eig[n] for n in G.nodes()],
})

centrality_df.sort_values("in_deg_cent", ascending=False).head()

Top 10 in-degree Centrality Scores: [('322', 0.2679324894514768), ('208', 0.2552742616033755), ('190', 0.2531645569620253), ('111', 0.22995780590717296), ('254', 0.22784810126582278), ('385', 0.22784810126582278), ('269', 0.22362869198312235), ('192', 0.22151898734177214), ('147', 0.20464135021097044), ('303', 0.20464135021097044)]
Top 10 out-degree Centrality Scores: [('367', 0.4430379746835443), ('322', 0.3312236286919831), ('393', 0.2341772151898734), ('71', 0.20464135021097044), ('399', 0.18776371308016876), ('436', 0.1793248945147679), ('179', 0.1772151898734177), ('254', 0.16666666666666666), ('105', 0.1582278481012658), ('87', 0.14978902953586495)]
----------------------------------------------------------------------------------------------------


,node,in_deg_cent,out_deg_cent,eigenvector
270,322,0.267932,0.331224,0.197757
313,208,0.255274,0.128692,0.090214
99,190,0.253165,0.078059,0.054380
78,111,0.229958,0.082278,0.048097
51,254,0.227848,0.166667,0.103206


#### Doing the same Centrality Metrics with the Weighted Graph that was Pulled using repo code.

In [9]:
G_w = nx.read_weighted_edgelist('network.weighted.edgelist', create_using=nx.DiGraph())
print("Loaded: 'network.weighted.edgelist'")
print("Directed:", G_w.is_directed())
print("Nodes:", G_w.number_of_nodes(), "Edges:", G_w.number_of_edges())

Loaded: 'network.weighted.edgelist'
Directed: True
Nodes: 27 Edges: 47


In [10]:
## Getting the needed metrics for the eventual centrality calculations 
density_w = nx.density(G_w)
print("Density:", density_w)

## Do we have any isolated nodes? 
isolated_nodes_w = list(nx.isolates(G_w))
print("Count of isolated nodes:", len(isolated_nodes_w))

## Looking at the nodes in the graph that are weakly connected. S

Density: 0.06695156695156695
Count of isolated nodes: 0


In [11]:
## Seeing how much of these connections are weakly connected.
wccs_w = list(nx.weakly_connected_components(G_w))
print("Weakly connected components:", len(wccs_w))

largest_wcc_w = max(wccs_w, key=len)
print("Largest WCC size:", len(largest_wcc_w))

Weakly connected components: 2
Largest WCC size: 22


In [12]:
## Calculcating the in / out degree strengths for for the G_w 
in_strength_w  = dict(G_w.in_degree(weight="weight"))
out_strength_w = dict(G_w.out_degree(weight="weight"))

In [ ]:
### By using the weights in the pulled network above, we will get a better sense of the true centrality of each node.
### With the edge weighted numbers, which were generated by the repo code, they highlight the proportion of total interaction from the sender 
### directed towards the examined node. The strength is quantified by summing the weighted edges yeilding a number indicative of interaction intensity.

### This means that when looking at in or out degree strengths, the numbers shift from total interactions with other nodes outwards/inwards
### towards how strongly the interactions are directed.

### In short, the weighted number gives dimension to the quality of the interactions vs just the quantity. 

In [13]:
eig_w = nx.eigenvector_centrality(G_w.reverse(), weight="weight", max_iter=5000, tol=1e-8)
# eig_w

In [ ]:
### Similarly, with the eigenvector centrality considering the weights, shift from simple connections to looking at the stronger and repeated
### interactions with the "important" notes, rather than considering all equally.  

In [14]:
### Getting the top 10 values dfor each of these metrics
print("Top 10 in-degree Centrality Scores for Weighted:", sorted(in_strength_w.items(), key=lambda x: x[1], reverse=True)[:10])
print("Top 10 out-degree Centrality Scores for Weighted:", sorted(out_strength_w.items(), key=lambda x: x[1], reverse=True)[:10])
print ('-'*100)

### Placing the values in a data fram
centrality_df_w = pd.DataFrame({
    "node": list(G_w.nodes()),
    "in_strength_w": [in_strength_w[n] for n in G_w.nodes()],
    "out_strength_w": [out_strength_w[n] for n in G_w.nodes()],
    "eigenvector": [eig_w[n] for n in G_w.nodes()],
})

centrality_df_w.sort_values("in_strength_w", ascending=False).head()

Top 10 in-degree Centrality Scores for Weighted: [('1067541214671577093', 0.04051552054517688), ('1067748650485497862', 0.026143790849673203), ('1071102246', 0.020438702995544823), ('1058717720', 0.019485364033996984), ('1069636653353000962', 0.01621591699126675), ('1067818539179024386', 0.015555555555555555), ('1071402577', 0.01450198404417516), ('1060370282', 0.014484126984126985), ('1061310112013434882', 0.013933174475919258), ('1017500185356853248', 0.011296111665004986)]
Top 10 out-degree Centrality Scores for Weighted: [('1037321378', 0.04060913705583756), ('1058717720', 0.032679738562091505), ('1067748650485497862', 0.017291066282420747), ('1052896620797460481', 0.0169971671388102), ('1069124515', 0.016666666666666666), ('1069636653353000962', 0.01486988847583643), ('1071840474', 0.01485148514851485), ('1051127714', 0.014234875444839857), ('1017819745880543238', 0.010135135135135136), ('1028854804087492613', 0.01)]
----------------------------------------------------------------

,node,in_strength_w,out_strength_w,eigenvector
7,1067541214671577093,0.040516,0.006098,2.107966e-06
16,1067748650485497862,0.026144,0.017291,5.833615e-01
6,1071102246,0.020439,0.003953,2.985308e-09
12,1058717720,0.019485,0.032680,7.857817e-01
13,1069636653353000962,0.016216,0.014870,3.869023e-06


### Taking a look at the results (Original & With weights) 

In [15]:
## Each of the nodes are individuals in the network, these centrality weights look at the interactions themselves, without looking
## at the repetitive nature (quality) of the interactions. While the weighted centrality df looks at the interactions and how they are sustained
## and how the engagement is concentrated via repetition.
print(centrality_df)
print(centrality_df_w)

    node  in_deg_cent  out_deg_cent  eigenvector
0      0     0.054852      0.042194     0.025788
1      4     0.063291      0.073840     0.042611
2     12     0.078059      0.052743     0.026116
3     18     0.052743      0.042194     0.018052
4     25     0.084388      0.086498     0.055183
..   ...          ...           ...          ...
470  227     0.000000      0.029536     0.022087
471  240     0.000000      0.010549     0.005112
472  356     0.000000      0.021097     0.011522
473  434     0.000000      0.010549     0.006445
474  456     0.000000      0.052743     0.033311

[475 rows x 4 columns]
                   node  in_strength_w  out_strength_w   eigenvector
0   1017500185356853248       0.011296        0.004225  5.763121e-09
1            1048784496       0.001408        0.007059  1.015840e-08
2            1058520120       0.010424        0.000000  4.639174e-11
3            1071402577       0.014502        0.004237  6.116451e-09
4   1017819745880543238       0.008801     

### Enriching the results with Handles. 

In [16]:
# Look at a few node labels from each graph
print("Unweighted sample nodes:", list(G.nodes())[:10])
print("Weighted sample nodes:", list(G_w.nodes())[:10])

# Are they digits / ints / usernames?
print("Unweighted node types:", {type(n) for n in list(G.nodes())[:10]})
print("Weighted node types:", {type(n) for n in list(G_w.nodes())[:10]})

Unweighted sample nodes: ['0', '4', '12', '18', '25', '30', '46', '55', '58', '59']
Weighted sample nodes: ['1017500185356853248', '1048784496', '1058520120', '1071402577', '1017819745880543238', '1051127714', '1071102246', '1067541214671577093', '1052896620797460481', '1061310112013434882']
Unweighted node types: {<class 'str'>}
Weighted node types: {<class 'str'>}


In [20]:
### Reading  in the usernames from the repo file
with open("congress_network_data.json", "r") as f:
    cdata = json.load(f)
cdata[0].keys()

dict_keys(['inList', 'inWeight', 'outList', 'outWeight', 'usernameList'])

In [21]:
# Mapping these to the unweighted full raw network
username_list = cdata[0]["usernameList"]
idx_to_handle = {
    str(i): username_list[i].lstrip("@").lower()
    for i in range(len(username_list))
}
centrality_df["handle"] = centrality_df["node"].astype(str).map(idx_to_handle)

In [22]:
print(centrality_df["handle"].isna().mean())
centrality_df[centrality_df["handle"].isna()].head()

0.0


,node,in_deg_cent,out_deg_cent,eigenvector,handle


In [23]:
## Reading in the congressional ids fro mthe repo file, and maping those to the handles in roder to put with sub graph with weights.
with open("congress_ids.txt", "r") as f:
    id_list = [line.strip() for line in f if line.strip()]
id_to_handle = {
    id_list[i]: username_list[i].lstrip("@").lower()
    for i in range(min(len(id_list), len(username_list)))
}

In [24]:
centrality_df_w["handle"] = centrality_df_w["node"].astype(str).map(id_to_handle)

In [25]:
print(centrality_df_w["handle"].isna().mean())
centrality_df_w[centrality_df_w["handle"].isna()][["node"]].head(20)
### NOTE: there are 4 unmapped node id's, continuing on anyway. 

0.14814814814814814


,node
4,1017819745880543238
10,1037321378
13,1069636653353000962
26,1051446626


### Looking at Results for both DFs

In [26]:
### Pulling the chamber information for each based on trhe handle name found. This is the category for this project. 

def infer_chamber_from_handle(h):
    if pd.isna(h):
        return np.nan
    h = str(h).lower()
    if h.startswith("sen") or h.startswith("senator"):
        return "Senate"
    if h.startswith("rep") or h.startswith("representative"):
        return "House"
    return "Other/Unknown"  # keep but you can exclude from grouped analysis if you want

centrality_df["chamber"] = centrality_df["handle"].apply(infer_chamber_from_handle)
centrality_df_w["chamber"] = centrality_df_w["handle"].apply(infer_chamber_from_handle)

centrality_df["chamber"].value_counts(dropna=False), centrality_df_w["chamber"].value_counts(dropna=False)

(chamber
 House            318
 Other/Unknown     90
 Senate            67
 Name: count, dtype: int64,
 chamber
 House            14
 Other/Unknown     5
 Senate            4
 NaN               4
 Name: count, dtype: int64)

In [27]:
### unqweigfhted vs the weighted sub graph 
df_u = centrality_df.dropna(subset=["chamber"]).copy()
df_w = centrality_df_w.dropna(subset=["chamber"]).copy()
## For simplicity were dropping those that are not in either chamber by our methods using handles.
df_u = df_u[df_u["chamber"].isin(["House", "Senate"])]
df_w = df_w[df_w["chamber"].isin(["House", "Senate"])]


In [31]:
summary_u = df_u.groupby("chamber")[["in_deg_cent", "out_deg_cent", "eigenvector"]].agg(["count", "mean", "median", "std"]).round(4)
summary_u

in_deg_cent                         out_deg_cent                  \
              count    mean  median     std        count    mean  median   
chamber                                                                    
House           318  0.0625  0.0464  0.0494          318  0.0575  0.0506   
Senate           67  0.0482  0.0464  0.0266           67  0.0631  0.0570   

                eigenvector                          
            std       count    mean  median     std  
chamber                                              
House    0.0315         318  0.0390  0.0349  0.0212  
Senate   0.0307          67  0.0359  0.0319  0.0200

In [81]:
### For the unweighted data we are left with 318 house members and 67 different senators in our network. This breakdown shows that:

### - Members of the house have higher in-degfree centrality on average. This can be interpreted to mean they are contacted by more of 
###   their peers on avg. However, there is a larger standard deviation, which shows the discrepancy between house members centrality is larger than 
###   that senators. 

### - Senators have a higher mean out-degree of centrality, which makes sense. As this imploies that senators outwardly
###   with more of their peers on average than house members. 

### - The avg. eigenvector centralities between represenatativs and senators is similar. The numbers are very close, but the house has a higher value
###   that may be indicative of a higher influence within the netwrok, but again the variablility in the house is a bit higher than in the senate. 
###   Meaning this influence is not as equitably distributed in the chamber. 

In [32]:
summary_w = df_w.groupby("chamber")[["in_strength_w", "out_strength_w", "eigenvector"]].agg(
    ["count", "mean", "median", "std"]
).round(4)

summary_w

in_strength_w                         out_strength_w                  \
                count    mean  median     std          count    mean  median   
chamber                                                                        
House              14  0.0124  0.0108  0.0117             14  0.0096  0.0051   
Senate              4  0.0035  0.0017  0.0047              4  0.0054  0.0069   

                eigenvector                         
            std       count    mean median     std  
chamber                                             
House    0.0095          14  0.0978    0.0  0.2517  
Senate   0.0037           4  0.0000    0.0  0.0000

In [ ]:
### For the weighted data we are left with 14 house members and 4 different senators in our network. This breakdown shows that:

### - Members ofthe house again have a higher in-strength on average.  This can be interpreted to mean they are contacted by more of 
###   their peers on avg. Again, there is a larger standard deviation, which shows the discrepancy between house members centrality is larger than 
###   that of senators. 

### - In the weighted version of this, the house also has a higher level of out-strength than the senators in this subset. This they are
###   also contacting more of their peers than the senators in this group on avg. The standard deviation is also higher, so this varies more between 
###  representatives than senators.

### - The most drastic difference found here is the mean eigenvector centrality for those in the house and those in the senate for the weighted subgraph.
###  The mean eigenvector centrality also varies much more in the house reps thn the senators. That being said, the senator sample is super small, which
### could be impacting these numbers. 

In [ ]:
### When looking at the results across categories / groups in the unweighted and the weighted centralities:

###  The metrics calculated on the unweighted raw network obtains the general patterns of engagement across the sample, while the weighted
### subset graph shows that these metrics can vary widely amongst different members/nodes depending on the weights within, the and the sample itself.
